# Tester GLiNER sur une phrase

Ce notebook exécute uniquement la branche GLiNER sur une phrase. Il reprend le modèle, le seuil et les groupes de labels du notebook `03e_tester_comparaison_branches.ipynb`.

Chaque groupe est envoyé séparément à GLiNER, exactement comme dans `analyze_gliner_section`.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "compliance_nlp").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet.")


ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"
GLINER_MODEL = rf"{MODEL_STORE_DIR}\gliner_multi-v2.1"
GLINER_SOURCE_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LOCAL_FILES_ONLY = True

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

pd.set_option("display.max_colwidth", 180)

In [ ]:
# Paramétrage identique au notebook 03e.
GLINER_LABEL_GROUPS = {
    "sante": [
        "donnee de sante",
        "maladie",
        "pathologie",
        "autisme",
        "handicap",
        "trouble du neurodeveloppement",
        "etat de sante",
        "condition medicale",
        "probleme de sante",
    ],
    "politique": [
        "ideologie politique",
        "position politique",
    ],
    "religion": ["conviction religieuse", "religion", "appartenance religieuse"],
    "syndical": ["appartenance syndicale", "engagement syndical"],
    "orientation_sexuelle": ["orientation sexuelle"],
    "origine": ["origine ethnique", "origine raciale"],
    "biometrie_genetique": ["donnee genetique", "donnee biometrique"],
    "conformite_conseil": [
        "clause beneficiaire imprecise",
        "conseil non professionnel",
        "promesse de performance",
    ],
}

pd.DataFrame(
    [
        {"groupe": groupe, "labels": ", ".join(labels)}
        for groupe, labels in GLINER_LABEL_GROUPS.items()
    ]
)

## Chargement du modèle

Le modèle est chargé depuis le même emplacement local que dans le notebook 03e. Sur CPU, le premier chargement peut prendre environ une minute.

In [ ]:
import gliner
from compliance_nlp.gliner_detector import load_gliner_model

model = load_gliner_model(
    model_name=GLINER_MODEL,
    cache_dir=MODEL_CACHE_DIR,
    source_model=GLINER_SOURCE_MODEL,
    local_files_only=GLINER_LOCAL_FILES_ONLY,
)
model.eval()

pd.Series({
    "version_gliner": getattr(gliner, "__version__", "inconnue"),
    "modele": GLINER_MODEL,
    "seuil": GLINER_THRESHOLD,
    "mode_evaluation": not model.training,
})

## Phrase à tester

Modifiez uniquement `PHRASE`. La chaîne complète est transmise à GLiNER pour chaque groupe de labels.

In [ ]:
PHRASE = "RDV mme et sa fille pour sucession du papa.CP"

print(repr(PHRASE))

In [ ]:
RESULT_COLUMNS = [
    "groupe", "texte_detecte", "label", "score",
    "debut", "fin", "seuil_applique",
]


def tester_gliner_par_groupes(phrase: str, threshold: float) -> pd.DataFrame:
    rows = []
    for groupe, labels in GLINER_LABEL_GROUPS.items():
        entities = model.predict_entities(
            phrase,
            list(labels),
            threshold=threshold,
        )
        for entity in entities:
            rows.append({
                "groupe": groupe,
                "texte_detecte": entity["text"],
                "label": entity["label"],
                "score": float(entity["score"]),
                "debut": entity["start"],
                "fin": entity["end"],
                "seuil_applique": threshold,
            })
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

## Résultats retenus avec le seuil de production

Cette cellule reproduit le comportement du notebook 03e avec `GLINER_THRESHOLD = 0.50`. Une table vide signifie qu'aucune alerte GLiNER n'est retenue.

In [ ]:
resultats_retenus = tester_gliner_par_groupes(PHRASE, GLINER_THRESHOLD)
resultats_retenus.sort_values("score", ascending=False, ignore_index=True)

## Diagnostic des scores bruts

Le seuil `0.0` permet d'observer les rapprochements du modèle même lorsqu'ils ne déclenchent pas d'alerte. Ce seuil sert uniquement au diagnostic.

In [ ]:
resultats_bruts = tester_gliner_par_groupes(PHRASE, threshold=0.0)
resultats_bruts.sort_values("score", ascending=False, ignore_index=True)

## Analyse complète de la phrase

GLiNER analyse la phrase entière, puis restitue les passages qu'il associe aux labels. La table suivante rassemble tous ces passages sans filtrage sur `CP` ou sur un autre mot.

In [ ]:
analyse_complete = resultats_bruts.copy()
analyse_complete.insert(0, "phrase_analysee", PHRASE)

analyse_complete["alerte_au_seuil_production"] = (
    analyse_complete["score"] >= GLINER_THRESHOLD
)
analyse_complete.sort_values("score", ascending=False, ignore_index=True)